# Lesson 4: Your First ReAct Agent

## WHY
In Lesson 3, you wrote the tool execution loop by hand — check for tool calls, run them, feed results back, repeat.
That works, but it's tedious and error-prone. What if the model calls two tools? What if it needs three rounds of back-and-forth?

The **ReAct** pattern (Reason + Act) automates this loop:
1. The model **reasons** about what to do next
2. It **acts** by calling a tool
3. It **observes** the tool result
4. Repeat until done

LangGraph's `create_react_agent` wraps this entire pattern in a single function call. This is the foundation of every agent in this curriculum.

**By the end of this notebook you will:**
1. Understand the ReAct reasoning pattern and why it works
2. Build an agent with `create_react_agent` in one line
3. Trace the agent's reasoning steps to understand its decisions
4. Configure agent behavior with system prompts
5. Handle edge cases: tool errors, max iterations, and model hallucinations
6. Build a complete moderation agent using the tools from Lesson 3
7. Package the agent as an importable factory function

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
print("API key loaded:", "OPENAI_API_KEY" in os.environ)

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## WHAT — The ReAct Pattern

ReAct stands for **Re**asoning + **Act**ing. It was introduced in a [2022 paper](https://arxiv.org/abs/2210.03629) that showed LLMs perform better when they interleave thinking and tool use.

The loop looks like this:

```
User: "Is this spam? 'FREE NITRO CLICK HERE'"
         ↓
Agent THINKS: "I should check the spam rules first"
Agent ACTS:   get_server_rules(category="spam")
Agent OBSERVES: "1. No self-promo, 2. No phishing..."
         ↓
Agent THINKS: "Now let me check the user's history"
Agent ACTS:   get_user_warn_history(user_id="user_123")
Agent OBSERVES: "2 previous warnings"
         ↓
Agent THINKS: "Repeat offender + clear spam. Issue mute."
Agent ACTS:   issue_moderation_action(...)
Agent OBSERVES: "[SIM] MUTE on user_123"
         ↓
Agent RESPONDS: "Muted user_123 for 30 min due to spam..."
```

Each Think → Act → Observe cycle is one **iteration** of the agent loop.

## HOW — `create_react_agent`

`create_react_agent` from LangGraph takes a model and a list of tools and returns a fully functional agent.
Under the hood it builds a **LangGraph StateGraph** (which you'll learn about in Module 5), but for now you can treat it as a black box that automates the tool loop.

```python
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(model, tools)
result = agent.invoke({"messages": [...]})
```

That's it. The agent handles:
- Binding tools to the model
- Detecting tool calls in the response
- Running the tools
- Feeding results back
- Looping until the model gives a final answer

📖 [create_react_agent reference](https://langchain-ai.github.io/langgraph/reference/prebuilt/#langgraph.prebuilt.chat_agent_executor.create_react_agent)

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool


# Let's start with simple tools to see the pattern clearly
@tool
def add(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b


# Create the agent — one line
agent = create_react_agent(llm, [add, multiply])

print(f"Type: {type(agent)}")
print("Agent created successfully!")

In [ ]:
from langchain_core.messages import HumanMessage

# Invoke the agent
result = agent.invoke({
    "messages": [HumanMessage(content="What is 15 + 27, then multiply the result by 3?")]
})

# The result contains the full message history
print(f"Total messages exchanged: {len(result['messages'])}\n")

# Print the final answer
print("Final answer:")
print(result["messages"][-1].content)

## HOW — Tracing the Agent's Steps

One of the most important skills in AI engineering is understanding *why* an agent did what it did.
The result object contains the full conversation history — every thought, tool call, and result.

In [ ]:
# Let's trace through every message in the agent's history
for i, msg in enumerate(result["messages"]):
    role = msg.__class__.__name__

    if hasattr(msg, "tool_calls") and msg.tool_calls:
        # AI message with tool calls
        for tc in msg.tool_calls:
            print(f"[{i}] {role} → TOOL CALL: {tc['name']}({tc['args']})")
    elif role == "ToolMessage":
        print(f"[{i}] {role} → Result: {msg.content}")
    else:
        content = msg.content[:100] + "..." if len(msg.content) > 100 else msg.content
        print(f"[{i}] {role} → {content}")

### Reading the Trace

You should see something like:

```
[0] HumanMessage → What is 15 + 27, then multiply the result by 3?
[1] AIMessage    → TOOL CALL: add({a: 15, b: 27})
[2] ToolMessage  → Result: 42
[3] AIMessage    → TOOL CALL: multiply({a: 42, b: 3})
[4] ToolMessage  → Result: 126
[5] AIMessage    → The result is 126.
```

Notice the agent:
1. Broke a multi-step problem into individual tool calls
2. Used the **output** of `add` as the **input** to `multiply`
3. Stopped calling tools once it had the final answer

This is the ReAct loop in action.

## HOW — Streaming Agent Steps

`.invoke()` waits for the entire agent loop to finish. For a Discord bot, you might want to see each step as it happens.
`.stream()` yields events after each node in the graph processes.

In [ ]:
# .stream() yields state updates after each step
for step in agent.stream(
    {"messages": [HumanMessage(content="What is 7 + 5, then multiply by 10?")]}
):
    # Each step has a key indicating which node produced it
    for node_name, node_output in step.items():
        print(f"--- Node: {node_name} ---")
        for msg in node_output["messages"]:
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  Tool call: {tc['name']}({tc['args']})")
            elif msg.__class__.__name__ == "ToolMessage":
                print(f"  Tool result: {msg.content}")
            else:
                print(f"  {msg.content[:120]}")
        print()

### Stream Output Explained

You'll see two node names alternating:
- **`agent`** — The LLM reasoning step (may produce tool calls or a final answer)
- **`tools`** — The tool execution step (runs the requested tools)

This agent/tools alternation IS the ReAct loop. In Module 5, you'll learn to build custom graphs with your own nodes — but `create_react_agent` is built on the same underlying `StateGraph`.

## HOW — System Prompts (Controlling Agent Behavior)

`create_react_agent` accepts a `prompt` parameter that sets the system message.
This is how you tell the agent *who it is* and *how it should behave* — critical for moderation.

In [ ]:
# Agent with a system prompt
moderation_system_prompt = (
    "You are a Discord server moderation bot.\n"
    "Your job is to evaluate messages for rule violations and take appropriate action.\n\n"
    "WORKFLOW:\n"
    "1. First check the relevant server rules\n"
    "2. Check the user's warning history\n"
    "3. Decide on an action (allow, warn, mute, kick, ban)\n"
    "4. Issue the action if needed\n\n"
    "GUIDELINES:\n"
    "- First offense + minor violation → warn\n"
    "- Repeat offender + minor violation → mute (30 min)\n"
    "- Severe violation (phishing, threats) → ban regardless of history\n"
    "- Always explain your reasoning"
)

print(moderation_system_prompt)

## HOW — Moderation Agent with Real Tools

Now let's combine the moderation tools from Lesson 3 with `create_react_agent`.
We'll redefine the tools here so this notebook is self-contained.

In [ ]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field


# Simulated data stores
SERVER_RULES = {
    "general": ["Be respectful", "Stay on-topic", "No excessive caps"],
    "spam": ["No self-promo", "No repeated messages", "No phishing links"],
    "nsfw": ["No NSFW outside designated channels", "No gore or shock content"],
}

WARN_HISTORY = {
    "user_123": [
        {"reason": "Spam links in #general", "date": "2026-03-15"},
        {"reason": "Harassment", "date": "2026-04-01"},
    ],
    "user_456": [{"reason": "NSFW in wrong channel", "date": "2026-02-20"}],
    "user_789": [],
}


@tool
def get_server_rules(category: str = "all") -> str:
    """Look up the Discord server's moderation rules.

    Use when you need to check what rules apply before making a moderation decision.

    Parameters
    ----------
    category : str
        Rule category: 'general', 'spam', 'nsfw', or 'all'.
    """
    if category == "all":
        return "\n".join(f"{c}: {', '.join(r)}" for c, r in SERVER_RULES.items())
    rules = SERVER_RULES.get(category)
    if rules:
        return "\n".join(f"{i}. {r}" for i, r in enumerate(rules, 1))
    return f"Unknown category: {category}. Available: {list(SERVER_RULES.keys())}"


class UserLookupInput(BaseModel):
    """Input for user history lookup."""
    user_id: str = Field(description="The Discord user's ID")


@tool(args_schema=UserLookupInput)
def get_user_warn_history(user_id: str) -> str:
    """Look up a Discord user's previous warnings and infractions.

    Check history before deciding on action. Repeat offenders need stricter action.
    """
    if user_id not in WARN_HISTORY:
        return f"No record found for user {user_id}"
    warnings = WARN_HISTORY[user_id]
    if not warnings:
        return f"User {user_id} has a clean record (0 warnings)"
    lines = [f"User {user_id}: {len(warnings)} warning(s)"]
    for w in warnings:
        lines.append(f"  - {w['date']}: {w['reason']}")
    return "\n".join(lines)


class ModActionInput(BaseModel):
    """Input for moderation actions."""
    user_id: str = Field(description="Discord user ID")
    action: str = Field(description="Action: 'warn', 'mute', 'kick', or 'ban'")
    reason: str = Field(description="Reason for the action")
    duration_minutes: int = Field(default=0, description="Duration for temp actions. 0 = permanent.")


@tool(args_schema=ModActionInput)
def issue_moderation_action(
    user_id: str, action: str, reason: str, duration_minutes: int = 0
) -> str:
    """Issue a moderation action against a Discord user.

    IMPORTANT: Always check rules and user history BEFORE calling this.
    Simulated in dev — calls Discord API in production.
    """
    valid = ["warn", "mute", "kick", "ban"]
    if action not in valid:
        return f"Invalid action. Must be one of: {valid}"
    dur = f" for {duration_minutes}m" if duration_minutes else ""
    return f"[SIM] {action.upper()}{dur} on {user_id}: {reason}"


moderation_tools = [get_server_rules, get_user_warn_history, issue_moderation_action]
print(f"Tools defined: {[t.name for t in moderation_tools]}")

In [ ]:
from langgraph.prebuilt import create_react_agent

# Create the moderation agent with system prompt
moderation_agent = create_react_agent(
    llm,
    moderation_tools,
    prompt=moderation_system_prompt,
)

print("Moderation agent created!")
print(f"Type: {type(moderation_agent)}")

In [ ]:
from langchain_core.messages import HumanMessage

# Test: Known repeat offender + spam
result = moderation_agent.invoke({
    "messages": [
        HumanMessage(
            content=(
                "User user_123 posted in #general: "
                "'FREE DISCORD NITRO! Click: totally-not-a-scam.com'\n"
                "Evaluate and take action."
            )
        )
    ]
})

# Print the full trace
print("=== Agent Trace ===")
for i, msg in enumerate(result["messages"]):
    role = msg.__class__.__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[{i}] {role} → TOOL: {tc['name']}({tc['args']})")
    elif role == "ToolMessage":
        print(f"[{i}] {role} → {msg.content[:100]}")
    elif role == "SystemMessage":
        print(f"[{i}] {role} → (system prompt)")
    else:
        print(f"[{i}] {role} → {msg.content[:120]}")

print("\n=== Final Verdict ===")
print(result["messages"][-1].content)

In [ ]:
# Test: Clean user + mild message
result2 = moderation_agent.invoke({
    "messages": [
        HumanMessage(
            content=(
                "User user_789 posted in #general: "
                "'Hey can someone help me with a Python question?'\n"
                "Evaluate this message."
            )
        )
    ]
})

print("=== Agent Trace ===")
for i, msg in enumerate(result2["messages"]):
    role = msg.__class__.__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[{i}] {role} → TOOL: {tc['name']}({tc['args']})")
    elif role == "ToolMessage":
        print(f"[{i}] {role} → {msg.content[:100]}")
    elif role == "SystemMessage":
        print(f"[{i}] {role} → (system prompt)")
    else:
        print(f"[{i}] {role} → {msg.content[:120]}")

print("\n=== Final Verdict ===")
print(result2["messages"][-1].content)

## Deep Dive — Agent Internals

`create_react_agent` builds a LangGraph `StateGraph` with two nodes:

| Node | What it does |
|------|--------------|
| `agent` | Calls the LLM (with tools bound). If the LLM returns tool calls → go to `tools` node. Otherwise → END. |
| `tools` | Executes each tool call, wraps results as `ToolMessage`, then → back to `agent` node. |

```
 START → agent → (tool calls?) → tools → agent → (tool calls?) → ... → END
              ↘ (no tool calls) → END
```

The **state** is simply the list of messages, which grows with each cycle.
In Module 5, you'll build your own `StateGraph` from scratch — but for now, this graph does exactly what we need.

## HOW — Handling Edge Cases

Agents can go wrong in several ways. Let's explore the important ones and how to handle them.

### Edge Case 1: Runaway Agents (Max Iterations)

If the model keeps calling tools in a loop, you need a safety limit.
`create_react_agent` accepts a `recursion_limit` parameter (via the graph config) that caps the number of steps.

In [ ]:
# Set a recursion limit to prevent infinite loops
config = {"recursion_limit": 10}  # max 10 graph steps (each agent→tools cycle = 2 steps)

result = moderation_agent.invoke(
    {"messages": [HumanMessage(content="Check user_123 and take action if needed for: 'spam spam spam'")]},
    config=config,
)

print(f"Total messages: {len(result['messages'])}")
print(f"Final answer: {result['messages'][-1].content[:150]}")

### Edge Case 2: Tool Errors

What happens when a tool raises an exception? Let's find out.

In [ ]:
@tool
def flaky_tool(query: str) -> str:
    """A tool that sometimes fails. Use this to search for information."""
    if "error" in query.lower():
        raise ValueError(f"Tool crashed on input: {query}")
    return f"Result for: {query}"


# Agent with the flaky tool
flaky_agent = create_react_agent(llm, [flaky_tool])

# This will trigger the error path
try:
    result = flaky_agent.invoke(
        {"messages": [HumanMessage(content="Search for: trigger error please")]}
    )
    # If the agent handles it, print the trace
    for msg in result["messages"]:
        role = msg.__class__.__name__
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            print(f"{role}: tool call → {msg.tool_calls[0]['name']}")
        else:
            content = msg.content[:100] if msg.content else "(empty)"
            print(f"{role}: {content}")
except Exception as e:
    print(f"Agent raised exception: {type(e).__name__}: {e}")
    print("\n→ In production, wrap agent.invoke() in try/except!")

### Edge Case 3: Model Doesn't Use Tools

Sometimes the model decides it can answer without tools — even when you'd prefer it use them.
The fix is a better system prompt (be more explicit about when tools are required).

In [ ]:
# Vague prompt — model may skip tools
vague_agent = create_react_agent(
    llm,
    moderation_tools,
    prompt="You moderate Discord messages.",
)

result = vague_agent.invoke({
    "messages": [HumanMessage(content="Is 'hello everyone' a rule violation?")]
})

tool_calls_made = sum(
    1 for m in result["messages"]
    if hasattr(m, "tool_calls") and m.tool_calls
)
print(f"Tool calls made: {tool_calls_made}")
print(f"Answer: {result['messages'][-1].content[:150]}")
print("\n→ Compare this with the detailed system prompt version")

## HOW — A Reusable Trace Printer

You'll trace agents constantly during development. Here's a helper to make it readable.

In [ ]:
def print_agent_trace(result: dict) -> None:
    """Print a readable trace of an agent's execution.

    Parameters
    ----------
    result : dict
        The result dict from agent.invoke(), containing a 'messages' key.
    """
    print("=" * 60)
    for i, msg in enumerate(result["messages"]):
        role = msg.__class__.__name__
        if role == "SystemMessage":
            print(f"[{i}] SYSTEM: (prompt omitted)")
        elif hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                args_str = ", ".join(f"{k}={v!r}" for k, v in tc["args"].items())
                print(f"[{i}] AGENT -> {tc['name']}({args_str})")
        elif role == "ToolMessage":
            content = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
            print(f"[{i}] TOOL  -> {content}")
        elif role == "HumanMessage":
            content = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
            print(f"[{i}] USER  -> {content}")
        else:
            content = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
            print(f"[{i}] AGENT -> {content}")
    print("=" * 60)


# Demo with a previous result
print_agent_trace(result2)

## Importable Agent Factory

Here's the moderation agent packaged as a factory function.
Tools are defined externally (as in Lesson 3) and passed in — keeping the agent flexible.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import BaseTool
from langgraph.prebuilt import create_react_agent
from langgraph.graph.state import CompiledStateGraph


DEFAULT_MODERATION_PROMPT = (
    "You are a Discord server moderation bot.\n"
    "Your job is to evaluate messages for rule violations and take appropriate action.\n\n"
    "WORKFLOW:\n"
    "1. Check the relevant server rules\n"
    "2. Check the user's warning history\n"
    "3. Decide on an appropriate action\n"
    "4. Issue the action if needed\n\n"
    "GUIDELINES:\n"
    "- First offense + minor violation → warn\n"
    "- Repeat offender + minor violation → mute (30 min)\n"
    "- Severe violation (phishing, threats) → ban regardless of history\n"
    "- Always explain your reasoning"
)


def create_moderation_agent(
    tools: list[BaseTool],
    model: str = "gpt-4o-mini",
    temperature: float = 0,
    system_prompt: str = DEFAULT_MODERATION_PROMPT,
) -> CompiledStateGraph:
    """Create a ReAct moderation agent.

    Parameters
    ----------
    tools : list[BaseTool]
        Moderation tools the agent can use.
    model : str
        OpenAI model name.
    temperature : float
        Sampling temperature. 0 = deterministic.
    system_prompt : str
        System instructions for the agent.

    Returns
    -------
    CompiledStateGraph
        A ready-to-use ReAct agent.
    """
    llm = ChatOpenAI(model=model, temperature=temperature)
    return create_react_agent(llm, tools, prompt=system_prompt)

In [ ]:
# Test the factory function
agent = create_moderation_agent(tools=moderation_tools)

result = agent.invoke({
    "messages": [
        HumanMessage(content="User user_456 said: 'Shut up loser, nobody asked'. Evaluate.")
    ]
})

print_agent_trace(result)
print("\n=== Final Verdict ===")
print(result["messages"][-1].content)

## Summary

| Concept | Key takeaway |
|---------|-------------|
| ReAct | Reason → Act → Observe loop; the LLM interleaves thinking with tool use |
| `create_react_agent` | One-line agent creation from LangGraph — handles the entire tool loop |
| Agent trace | Full message history shows every reasoning step — essential for debugging |
| `.stream()` | See agent/tools nodes fire in real-time |
| System prompt | Controls agent persona and workflow — be specific about when to use tools |
| `recursion_limit` | Safety cap on agent iterations |
| Error handling | Wrap `agent.invoke()` in try/except in production |
| Agent factory | Package agent creation as a function for clean import into bot code |

**Next up → Module 3, Lesson 5:** Structured output with Pydantic — making the agent return typed, validated moderation verdicts instead of free-text.